# 02 — Feature Engineering & the Degeneracy Ladder

Everything here calls `src.features` / `src.ablation` directly — this notebook adds no transformation logic of its own.

In [1]:
import pandas as pd
from src.features import build_and_split, FEATURE_SETS, DROP_COLS, TARGET_COL, feature_columns

train, test = build_and_split()
print(f'train: {train.shape}, test: {test.shape}')
train.head()

train: (48000, 36), test: (12000, 36)


,age,account_age_days,customer_segment,country,platform,device_type,payment_method,product_category,avg_order_value_usd,refund_amount_requested_usd,...,previous_dispute_count,wishlist_to_cart_time_hrs,review_left_after_return,abuse_type,refund_to_avg_order_ratio,returns_per_order,orders_kept_lifetime,orders_per_account_day,return_month,return_dayofweek
30083,18,1868,Silver,AU,Web Browser,iPad,Buy Now Pay Later,Toys,689.31,712.68,...,1,3.7,0,Fraudulent Return,1.033903,0.325000,27,0.021413,1,6
53526,38,291,Silver,US,Web Browser,iPhone,Debit Card,Jewelry,30.15,24.99,...,0,57.9,0,Legitimate,0.828856,0.137931,50,0.199313,1,6
3687,65,487,Platinum,US,Tablet App,iPad,Gift Card,Beauty,629.92,659.12,...,4,3.7,0,Fraudulent Return,1.046355,0.388889,11,0.036961,1,0
5918,35,1439,New,US,Tablet App,MacBook,Crypto,Furniture,751.27,732.58,...,4,0.8,0,Fraudulent Return,0.975122,0.571429,15,0.024322,1,0
27062,28,713,Bronze,DE,Web Browser,MacBook,Buy Now Pay Later,Shoes,335.85,351.54,...,1,2.4,0,Fraudulent Return,1.046717,0.619048,8,0.029453,1,0


## Dropped columns

`abuse_label` (1:1 target encoding), the two ID columns, and the two raw date columns (used to build the split + seasonality features, then dropped as raw strings).

In [2]:
DROP_COLS

['abuse_label', 'order_id', 'customer_id', 'order_date', 'return_date']

## Engineered features (§4.1)

`refund_to_avg_order_ratio`, `returns_per_order`, `orders_kept_lifetime`, `orders_per_account_day`, `return_month`, `return_dayofweek` — all built by `add_transaction_level_features`, the one function both the pipeline and `src/infer.py` call at serving time.

In [3]:
engineered = ['refund_to_avg_order_ratio', 'returns_per_order', 'orders_kept_lifetime',
              'orders_per_account_day', 'return_month', 'return_dayofweek']
train[engineered].describe()

,refund_to_avg_order_ratio,returns_per_order,orders_kept_lifetime,orders_per_account_day,return_month,return_dayofweek
count,48000.000000,48000.000000,48000.000000,48000.000000,48000.000000,48000.000000
mean,0.920056,0.190419,32.418896,0.140271,5.714104,3.006479
std,0.061057,0.227591,21.036357,1.123664,3.181456,1.992836
min,0.799990,0.000000,1.000000,0.000400,1.000000,0.000000
25%,0.870581,0.026667,14.000000,0.016445,3.000000,1.000000
50%,0.927656,0.086957,29.000000,0.032432,5.000000,3.000000
75%,0.969672,0.352941,49.000000,0.066305,8.000000,5.000000
max,1.049970,0.847059,80.000000,71.000000,12.000000,6.000000


## Dual-track feature sets

`full` uses every legitimate feature. `testbed` additionally drops the features the leakage finding implicates, *and their derived proxies* — dropping only the raw column and keeping an algebraic restatement of it measures nothing (see `docs/LEAKAGE_FINDING.md`'s ablation note).

In [4]:
print('full feature count:   ', len(feature_columns(train, track='full')))
print('testbed feature count:', len(feature_columns(train, track='testbed')))
print('testbed excludes:', FEATURE_SETS['testbed'])

full feature count:    35
testbed feature count: 27
testbed excludes: ['wishlist_to_cart_time_hrs', 'days_to_return', 'return_rate_pct', 'total_returns_lifetime', 'customer_support_contacts', 'previous_dispute_count', 'returns_per_order', 'orders_kept_lifetime']


## The degeneracy ladder

`src.ablation.run_ladder` drops the leakage-implicated features (and proxies) one group at a time and re-scores. This is diagnostic evidence about how degenerate the task is at each feature count — **not** a feature-selection procedure; nothing downstream is chosen by ablation performance.

In [5]:
from src.ablation import run_ladder
ladder = run_ladder(train, test)
ladder[['feature_set', 'n_features', 'macro_f1', 'threshold_sensitive_frac']]

A. all features                           35    0.9990      0.1%


B. -wishlist_to_cart_time_hrs             34    0.9936      0.2%


C. B -days_to_return                      33    0.9779      1.6%


D. C -return_rate_pct                     31    0.9767      2.0%


E. D -total_returns_lifetime              29    0.9586      3.7%


F. E -customer_support_contacts           28    0.9342      8.4%


G. F -previous_dispute_count  <- TESTBED  27    0.8993     16.1%


H. G -avg_order_value, -refund_amount     24    0.8581     23.9%


,feature_set,n_features,macro_f1,threshold_sensitive_frac
0,A. all features,35,0.999001,0.000667
1,B. -wishlist_to_cart_time_hrs,34,0.993616,0.002333
2,C. B -days_to_return,33,0.977900,0.016000
3,D. C -return_rate_pct,31,0.976674,0.019583
4,E. D -total_returns_lifetime,29,0.958623,0.037333
5,F. E -customer_support_contacts,28,0.934175,0.084333
6,G. F -previous_dispute_count <- TESTBED,27,0.899340,0.160667
7,"H. G -avg_order_value, -refund_amount",24,0.858110,0.239417


Macro-F1 degrades **smoothly** from 0.999 to 0.858 with no natural cut point — which is exactly why `testbed` (rung G) is chosen for one stated reason only (first rung with enough boundary mass for a cost sweep to move decisions), never because it scores well, and is never promoted to the headline model. Continued in `03_modeling.ipynb`.